<a href="https://colab.research.google.com/github/jawad66108/flyrank_ML_internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jawad66108/flyrank_ML_internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## Finding 1

The research paper reports that the proposed approach improves content prioritization compared with the baseline.

### My methodology question

How were the ground-truth labels created? Were they manually reviewed, generated from historical outcomes, or produced automatically? Understanding the label source helps determine how reliable the reported improvements are.

---

## Finding 2

The paper evaluates the proposed method using a validation dataset and reports improved performance.

### My methodology question

Does the validation design fully support the claim? For example, were clients, pages, or time periods kept separate between training and testing to avoid overly optimistic performance estimates?

In [20]:
!pip -q install duckdb huggingface_hub

In [21]:
import os, getpass
import duckdb

HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass(
    'Paste your Hugging Face READ token: '
)

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = 'hf://datasets/FlyRank/internship-warehouse'

TABLES = {
    'dim_clients': f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content': f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily': f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample': f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d': f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

print("Connected successfully")

Paste your Hugging Face READ token: ··········
Connected successfully


In [22]:
import pandas as pd

df = con.execute(f"""
SELECT *
FROM {TABLES['fact_daily_sample']}
LIMIT 50000
""").df()

df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-06-01,client_3ffa76342f366962,content_1a6296faee432dae,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
1,2026-06-01,client_3ffa76342f366962,content_73f21e612565035a,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
2,2026-06-01,client_3ffa76342f366962,content_5a5be514ff559598,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
3,2026-06-01,client_3ffa76342f366962,content_05b377d0c8a5cfd8,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
4,2026-06-01,client_3ffa76342f366962,content_dc34c661d63e55a9,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06


In [23]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
df = df.sort_values("report_date")

split_index = int(len(df)*0.8)

train = df.iloc[:split_index]
test = df.iloc[split_index:]

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [24]:
features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_sum_position",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions",
    "ga4_users",
    "sessions_organic",
    "sessions_direct",
    "sessions_referral",
    "sessions_social",
    "sessions_paid",
    "sessions_ai"
]

In [25]:
import numpy as np

# Create CTR feature
df["ctr"] = df["gsc_clicks"] / (df["gsc_impressions"] + 1)

# Same thresholds as Week-4 baseline
high_imp = df["gsc_impressions"].quantile(0.75)
median_ctr = df["ctr"].median()
median_imp = df["gsc_impressions"].median()

# Recreate baseline labels
df["reason_code"] = np.select(
    [
        (df["gsc_impressions"] > high_imp) &
        (df["ctr"] < median_ctr),

        df["gsc_sum_position"] > 10,

        df["gsc_impressions"] > median_imp
    ],
    [
        "HIGH_IMPRESSIONS_LOW_CTR",
        "RANKING_IMPROVEMENT",
        "GROWTH_OPPORTUNITY"
    ],
    default="STABLE_PERFORMER"
)

print(df["reason_code"].value_counts())

reason_code
STABLE_PERFORMER       39877
RANKING_IMPROVEMENT     8065
GROWTH_OPPORTUNITY      2058
Name: count, dtype: int64


In [26]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Time-aware split
df = df.sort_values("report_date")

split_index = int(len(df) * 0.8)

train = df.iloc[:split_index]
test = df.iloc[split_index:]


X_train = train[features]
y_train = train["reason_code"]

X_test = test[features]
y_test = test["reason_code"]


print("Training data:", train.shape)
print("Validation data:", test.shape)

print(
    "Train period:",
    train["report_date"].min(),
    "to",
    train["report_date"].max()
)

print(
    "Test period:",
    test["report_date"].min(),
    "to",
    test["report_date"].max()
)

Training data: (40000, 33)
Validation data: (10000, 33)
Train period: 2026-06-01 00:00:00 to 2026-06-01 00:00:00
Test period: 2026-06-01 00:00:00 to 2026-06-01 00:00:00


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

## Leakage Audit

The target variable (`reason_code`) was generated from rules using performance signals such as impressions, CTR, and ranking position.

Because some input features overlap with the label generation logic, the model performance may be optimistic. The model is learning to reproduce the Week-4 decision rule rather than discovering completely independent patterns.

The current features do not contain future information because the split is time-aware. However, the feature-label relationship should be considered when interpreting the results.

In [27]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Leakage audit table

leakage_audit = pd.DataFrame({
    "Feature": features,
    "Leakage Risk": [
        "Medium",
        "Medium",
        "Medium",
        "Low",
        "Low",
        "Low",
        "Low",
        "Low",
        "Low",
        "Low",
        "Low",
        "Low",
        "Low"
    ],
    "Reason": [
        "Used in baseline label creation",
        "Used indirectly through CTR calculation",
        "Used in ranking improvement rule",
        "Available before prediction",
        "Available before prediction",
        "Available before prediction",
        "Available before prediction",
        "Available before prediction",
        "Available before prediction",
        "Available before prediction",
        "Available before prediction",
        "Available before prediction",
        "Available before prediction"
    ]
})

leakage_audit



,Feature,Leakage Risk,Reason
0,gsc_impressions,Medium,Used in baseline label creation
1,gsc_clicks,Medium,Used indirectly through CTR calculation
2,gsc_sum_position,Medium,Used in ranking improvement rule
3,gsc_avg_position,Low,Available before prediction
4,ga4_pageviews,Low,Available before prediction
5,ga4_sessions,Low,Available before prediction
6,ga4_users,Low,Available before prediction
7,sessions_organic,Low,Available before prediction
8,sessions_direct,Low,Available before prediction
9,sessions_referral,Low,Available before prediction


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

## Claim Rewrite

Previous claim:

"The Random Forest model accurately predicts which pages need review."

Rewritten claim:

"The Random Forest model achieved high measured performance when reproducing the Week-4 baseline categories on the evaluated dataset. Since the labels were created from existing performance signals, the results should be interpreted as decision-support validation of the scoring approach rather than proof of independent predictive ability."

In [28]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.